#### Semi-Supervised Behavioral State Modeling with Dimensionality Reduction for Pediatric Digital Health Applications

#### Data Source

The data is obtained from the United States census bereau and is titled "National Survey for Childrens Health" for the year 2024. The data source Url: https://www.census.gov/programs-surveys/nsch/data/datasets.html 

#### Data Exploration ####

In [1]:
# Loading the SQL extension and connecting to the DuckDB database

%load_ext sql
%sql duckdb:///nsch_database.duckdb

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'duckdb:///nsch_database.duckdb'

Checking the current running database

In [2]:

%%sql
SELECT current_database();

Running query in 'duckdb:///nsch_database.duckdb'

current_database()
nsch_database


Checking the number of children enrolled in the survey

In [3]:
%%sql
SELECT (SELECT COUNT(*) FROM nsch_topical_2025) AS n_children,
       (SELECT COUNT(*) FROM information_schema.columns WHERE table_name = 'nsch_topical_2025') AS n_columns,
       (SELECT COUNT(*) FROM nsch_topical_2025_value_labels) AS n_value_label_rows

Running query in 'duckdb:///nsch_database.duckdb'

n_children,n_columns,n_value_label_rows
45139,458,3132


Checking the schema of the database

In [4]:
%%sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name

Running query in 'duckdb:///nsch_database.duckdb'

table_name
nsch_screener_2024
nsch_screener_2024_value_labels
nsch_screener_2024_var_labels
nsch_topical_2025
nsch_topical_2025_value_labels
nsch_topical_2025_var_labels


Summary statistics of the dataset

In [5]:
%%sql
SELECT * FROM (SUMMARIZE nsch_topical_2025)

Running query in 'duckdb:///nsch_database.duckdb'

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
height,DOUBLE,104.31,192.49,37,153.7257514985974,20.018880835027375,137.74492325460906,157.48,167.690333622741,45139,29.41
fipsst,BIGINT,1,56,47,28.333458871485856,16.029561657685242,16,28,42,45139,0.00
stratum,VARCHAR,1,2A,2,None,None,None,None,None,45139,0.00
hhid,BIGINT,25000005,25359999,43553,25179960.773721173,104127.0289747883,25089230,25180210,25269782,45139,0.00
formtype,VARCHAR,T1,T3,3,None,None,None,None,None,45139,0.00
totkids_r,BIGINT,1,4,4,1.820177673408804,0.8679403965920728,1,2,2,45139,0.00
tenure,BIGINT,1,4,4,1.4526019628259377,0.7756333629545974,1,1,2,45139,0.00
hhlanguage,INTEGER,1,3,3,1.1019363454262185,0.3987318681977409,1,1,1,45139,0.46
sc_age_years,BIGINT,0,17,20,9.71104809588161,5.231116067281949,5,10,14,45139,0.00
sc_sex,BIGINT,1,2,2,1.4813354305589401,0.49965704708367975,1,1,2,45139,0.00


Buidling the Cohort and target labels

In [6]:
%%sql
CREATE OR REPLACE TABLE nsch25_cohort AS
SELECT t.*,
       CASE WHEN k2q32a = 1 OR k2q33a = 1 OR k2q34a = 1 THEN 1
            WHEN k2q32a = 2 AND k2q33a = 2 AND k2q34a = 2 THEN 0
       END AS y
FROM nsch_topical_2025 t
WHERE sc_age_years BETWEEN 0 AND 17

Running query in 'duckdb:///nsch_database.duckdb'

Count


Checking the number of children who were asked questions about mental health

In [7]:
%%sql
SELECT COUNT(*) AS n_children,
       COUNT(y) AS n_with_label,
       COUNT(*) - COUNT(y) AS n_undetermined
FROM nsch25_cohort

Running query in 'duckdb:///nsch_database.duckdb'

n_children,n_with_label,n_undetermined
45139,44651,488


In [8]:
%%sql
SELECT y, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM nsch25_cohort GROUP BY y ORDER BY y

Running query in 'duckdb:///nsch_database.duckdb'

y,n,pct
0,36219,80.2
1,8432,18.7
None,488,1.1


Checking whether young children were actually asked these questions or were recorded as “No” by defaul

In [9]:
%%sql
SELECT CASE WHEN sc_age_years <= 2 THEN '0-2'
            WHEN sc_age_years <= 5 THEN '3-5'
            WHEN sc_age_years <= 11 THEN '6-11'
            ELSE '12-17' END AS age_band,
       COUNT(*) AS n,
       COUNT(y) AS n_with_label,
       ROUND(100.0 * AVG(y), 1) AS pct_positive,
       ROUND(100.0 * AVG(CASE WHEN k2q33a = 1 THEN 1 WHEN k2q33a = 2 THEN 0 END), 1) AS anxiety_pct,
       ROUND(100.0 * AVG(CASE WHEN k2q32a = 1 THEN 1 WHEN k2q32a = 2 THEN 0 END), 1) AS depression_pct
FROM nsch25_cohort
GROUP BY 1 ORDER BY 1

Running query in 'duckdb:///nsch_database.duckdb'

age_band,n,n_with_label,pct_positive,anxiety_pct,depression_pct
0-2,5556,5485,1.1,0.4,0.0
12-17,19691,19479,26.6,21.3,11.3
3-5,6544,6473,7.8,2.9,0.2
6-11,13348,13214,20.3,13.2,2.4


In [7]:
%%sql
SELECT sc_age_years,
       COUNT(*) AS n,
       COUNT(k2q32a) AS depression_answered,
       COUNT(k2q33a) AS anxiety_answered,
       COUNT(k2q34a) AS behavior_answered,
       ROUND(100.0 * AVG(CASE WHEN k2q33a = 1 THEN 1 WHEN k2q33a = 2 THEN 0 END), 1) AS anxiety_yes_pct
FROM nsch_topical_2025
GROUP BY sc_age_years ORDER BY sc_age_years

Running query in 'duckdb:///nsch_database.duckdb'

sc_age_years,n,depression_answered,anxiety_answered,behavior_answered,anxiety_yes_pct
0,1380,1369,1366,1376,0.3
1,1662,1654,1655,1659,0.4
2,2514,2504,2497,2511,0.4
3,2264,2243,2249,2260,1.3
4,2131,2123,2119,2128,2.6
5,2149,2141,2137,2142,4.9
6,2019,2005,1999,2018,7.6
7,2013,2006,2000,2011,9.4
8,2118,2098,2108,2111,12.5
9,2392,2381,2380,2391,14.1


In [8]:
%%sql
SELECT (SELECT COUNT(*) FROM nsch_topical_2024_value_labels) AS n_value_labels,
       (SELECT COUNT(*) FROM nsch_topical_2024_var_labels WHERE label <> '') AS n_var_labels

Running query in 'duckdb:///nsch_database.duckdb'

n_value_labels,n_var_labels
0,457


Checking that the variables exist in the dataset

In [9]:
%%sql
SELECT column_name FROM information_schema.columns
WHERE table_name = 'nsch_topical_2025'
  AND lower(column_name) IN ('sc_age_years','fwc','hhid','stratum','formtype','k2q32a','k2q33a','k2q34a')

Running query in 'duckdb:///nsch_database.duckdb'

column_name
stratum
hhid
formtype
sc_age_years
k2q33a
k2q32a
k2q34a
fwc


In [12]:
%%sql
CREATE OR REPLACE VIEW nsch_topical_2025_dictionary AS
SELECT c.column_name,
       c.data_type,
       l.label AS description,
       v.n_values,
       v.response_options
FROM information_schema.columns c
LEFT JOIN nsch_topical_2025_var_labels l USING (column_name)
LEFT JOIN (
    SELECT column_name,
           COUNT(*) AS n_values,
           string_agg(CAST(CAST(code AS BIGINT) AS VARCHAR) || '=' || label, '; ' ORDER BY code) AS response_options
    FROM nsch_topical_2025_value_labels
    GROUP BY column_name
) v USING (column_name)
WHERE c.table_name = 'nsch_topical_2025'

Running query in 'duckdb:///nsch_database.duckdb'

Count


In [13]:
%%sql
SELECT column_name, description, response_options
FROM nsch_topical_2025_dictionary
WHERE description ILIKE '%anxiety%' OR description ILIKE '%depress%'

Running query in 'duckdb:///nsch_database.duckdb'

column_name,description,response_options


In [11]:
%%sql
SELECT *
FROM nsch_topical_2024_dictionary limit 5;

Running query in 'duckdb:///nsch_database.duckdb'

column_name,data_type,description,n_values,response_options
height,DOUBLE,Child's Height (CM),4,None
fipsst,BIGINT,State FIPS Code,55,1=Alabama; 2=Alaska; 4=Arizona; 5=Arkansas; 6=California; 8=Colorado; 9=Connecticut; 10=Delaware; 11=District of Columbia; 12=Florida; 13=Georgia; 15=Hawaii; 16=Idaho; 17=Illinois; 18=Indiana; 19=Iowa; 20=Kansas; 21=Kentucky; 22=Louisiana; 23=Maine; 24=Maryland; 25=Massachusetts; 26=Michigan; 27=Minnesota; 28=Mississippi; 29=Missouri; 30=Montana; 31=Nebraska; 32=Nevada; 33=New Hampshire; 34=New Jersey; 35=New Mexico; 36=New York; 37=North Carolina; 38=North Dakota; 39=Ohio; 40=Oklahoma; 41=Oregon; 42=Pennsylvania; 44=Rhode Island; 45=South Carolina; 46=South Dakota; 47=Tennessee; 48=Texas; 49=Utah; 50=Vermont; 51=Virginia; 53=Washington; 54=West Virginia; 55=Wisconsin; 56=Wyoming
hhid,BIGINT,Unique Household ID,4,None
formtype,VARCHAR,Form Type,4,None
totkids_r,BIGINT,Number of Children in Household,8,1=1; 2=2; 3=3; 4=4+


In [7]:
%%sql
SELECT column_name FROM information_schema.columns
WHERE table_name = 'nsch_screener_2024' ORDER BY ordinal_position

Running query in 'duckdb:///nsch_database.duckdb'

column_name
fipsst
stratum
totkids_r
tenure
hhlanguage
year
cbsafp_yn
metro_yn
mpc_yn
tenure_if


In [8]:
%%sql
SELECT COUNT(*) AS shared_ids
FROM nsch_screener_2024 s
JOIN nsch_topical_2025 t ON s.hhid = t.hhid

Running query in 'duckdb:///nsch_database.duckdb'

RuntimeError: (_duckdb.BinderException) Binder Error: Table "s" does not have a column named "hhid"

Candidate bindings: : "hhids", "c_hispanic_r", "c_hispanic_r_if", "hhlanguage", "fwh"

LINE 3: JOIN nsch_topical_2025 t ON s.hhid = t.hhid
                                    ^
[SQL: SELECT COUNT(*) AS shared_ids
FROM nsch_screener_2024 s
JOIN nsch_topical_2025 t ON s.hhid = t.hhid]
(Background on this error at: https://sqlalche.me/e/21/f405)
